# Práctica de RAG — narrativa completa (naive → hybrid → advanced → capstone)

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase03/notebooks/rag/practica_completa.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Objetivo:** construir un sistema RAG sobre el programa real de la Cátedra IA UTN-FRSF 2026, pasarlo por un benchmark de 7 queries diversas, y agregar técnicas progresivamente hasta ver el contraste **completo** LLM-puro → RAG naive → hybrid → reranking → HyDE → parent-child.
>
> **Por qué este corpus:** el programa de la cátedra (docentes, ordenanzas, fechas, umbrales de aprobación) contiene datos **imposibles de adivinar** para el LLM. El contraste RAG-vs-LLM puro es dramático y mensurable.
>
> **Estructura:**
> - **Parte 1** — Setup + motivación (LLM puro vs RAG hardcoded)
> - **Parte 2** — Pipeline RAG naive completo (corpus → chunking → ChromaDB → retrieval → augmentation → generation)
> - **Parte 3** — Benchmark sobre el corpus + correr naive con side-by-side y visualización
> - **Parte 4** — Hybrid Search (BM25 + dense + RRF) y comparar 3 métodos
> - **Parte 5** — Técnicas avanzadas (Reranking + HyDE + Parent-Child) y comparar 5 modos
> - **Parte 6** — Capstone: una query difícil evoluciona por 6 stages
>
> **Tiempo:** ~90-110 min de hands-on en total (se puede partir en sesiones).


# Parte 1 — Setup + motivación

Antes de meter retrieval automático, veamos qué pasa cuando le preguntamos al LLM cosas que **no puede saber**.


## 0. Setup

Instalá dependencias y configurá la API key de Groq.


In [ ]:
!pip install -q sentence-transformers chromadb rank_bm25 groq numpy pandas matplotlib


In [ ]:
import os

try:
    from google.colab import userdata
    if 'GROQ_API_KEY' not in os.environ:
        try:
            os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
        except Exception:
            pass
except ImportError:
    pass

assert os.environ.get('GROQ_API_KEY'), (
    'Falta GROQ_API_KEY. Configurala en Colab Secrets o como env var.'
)
print('✓ GROQ_API_KEY configurada')


In [ ]:
from groq import Groq

_groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])


def llamar_llm(messages, model='llama-3.1-8b-instant', temperature=0.3, max_tokens=500):
    """Wrapper unificado para llamar Groq — el mismo de Clase 2."""
    resp = _groq_client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens,
    )
    return resp.choices[0].message.content


print(llamar_llm([{'role': 'user', 'content': 'Decime "ok" si me podés escuchar'}]))


In [ ]:
from sentence_transformers import SentenceTransformer

modelo_emb = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(f'✓ Modelo de embeddings: {modelo_emb.get_sentence_embedding_dimension()} dims')


## 1. Motivación — el LLM no puede saber esto

**Pregunta concreta:** *"¿Cuándo es el parcial de Inteligencia Artificial 2026 de la UTN-FRSF y qué tema cubre?"*

Esto **no está en el training del LLM**. Vamos a ver:
1. Qué responde el LLM puro — muy probablemente **inventa con confianza**.
2. Qué responde con contexto manual (RAG hardcoded simulado).

Después automatizamos la búsqueda del contexto.


In [ ]:
pregunta = '¿Cuándo es el parcial de Inteligencia Artificial 2026 de la UTN-FRSF y qué tema cubre?'

hechos_verdaderos = ['6 de abril', '06/04', 'Búsqueda en espacio de estado', 'campus virtual']


def score_keywords(respuesta, keywords):
    return sum(1 for k in keywords if k.lower() in respuesta.lower())


# ── 1) LLM PURO ──
resp_sin = llamar_llm([
    {'role': 'system', 'content': 'Respondé en español, máximo 4 oraciones.'},
    {'role': 'user', 'content': pregunta},
], temperature=0.3)
score_sin = score_keywords(resp_sin, hechos_verdaderos)

print('╔════════════════════════════════════════════════════════════╗')
print('║   LLM PURO (sin RAG)                                       ║')
print('╠════════════════════════════════════════════════════════════╣')
print(resp_sin)
print()
print(f'   ✗ Score de hechos correctos: {score_sin}/{len(hechos_verdaderos)}')
print('   ⚠️  Cualquier fecha o tema específico que mencione está')
print('       PROBABLEMENTE INVENTADO — el LLM no tiene esos datos.')
print('╚════════════════════════════════════════════════════════════╝')

# ── 2) CON contexto manual ──
contexto_manual = (
    'El parcial tiene fecha 6 de abril y cubre el tema "Búsqueda en espacio de estado", '
    'y se rinde en clase usando el campus virtual.'
)
resp_con = llamar_llm([
    {'role': 'system', 'content':
        'Respondé SOLO con base en el contexto. Si no alcanza, decilo. Máximo 4 oraciones.'},
    {'role': 'user', 'content': f'Contexto:\n{contexto_manual}\n\nPregunta: {pregunta}'},
], temperature=0.3)
score_con = score_keywords(resp_con, hechos_verdaderos)

print()
print('╔════════════════════════════════════════════════════════════╗')
print('║   CON contexto (RAG hardcoded)                             ║')
print('╠════════════════════════════════════════════════════════════╣')
print(resp_con)
print()
print(f'   ✓ Score de hechos correctos: {score_con}/{len(hechos_verdaderos)}')
print('   ✓ Información trazable al contexto provisto.')
print('╚════════════════════════════════════════════════════════════╝')

print()
print(f'💡 Diferencia visible: {score_con} vs {score_sin}. El RAG no es "más inteligente" —')
print('   tiene acceso a información que el LLM literalmente no puede saber.')
print('   Ahora vamos a automatizar la búsqueda del contexto.')


# Parte 2 — Pipeline RAG naive completo

Reemplazamos el "contexto manual" del paso anterior por un sistema automático:
**corpus → chunking → embeddings → vector store → retrieval → augmentation → generation**.


## 2. El corpus — programa real de la Cátedra IA 2026

5 documentos sobre el programa de la materia. Los datos son específicos: docentes con cargo, fechas, ordenanzas, umbrales, aula. **El LLM puro no puede inventar nada de esto correctamente.**


In [ ]:
# ── Corpus: Cátedra Inteligencia Artificial — UTN-FRSF, 1er cuatrimestre 2026 ──

DOCUMENTOS = [
    {
        "id": "doc_identidad",
        "titulo": "Identidad y modalidad de la cátedra",
        "contenido": (
            "La asignatura Inteligencia Artificial pertenece al 5to año de la carrera "
            "Ingeniería en Sistemas de Información (ISI) de la Universidad Tecnológica "
            "Nacional, Facultad Regional Santa Fe (UTN-FRSF), y se dicta durante el "
            "1er cuatrimestre del ciclo 2026. La cátedra está radicada en el CIDISI "
            "(Centro de Investigación de Desarrollo e Innovación en Sistemas de "
            "Información). Los docentes a cargo son la Profesora Titular Milagros "
            "Gutiérrez y el Profesor Adjunto Jorge Roa. La cátedra no cuenta con "
            "auxiliares ni ayudantes alumnos en esta cohorte. La materia se dicta "
            "de manera presencial los días lunes de 18:00 a 21:00 horas en el "
            "Laboratorio 1 de la facultad. La modalidad de cursado es por proyecto, "
            "con foco en aplicación práctica más que en exposición magistral. La "
            "cátedra opera bajo el marco normativo de la Ordenanza 1877, que "
            "establece las competencias específicas (CE) y genéricas (CG) requeridas "
            "para todas las ingenierías de la UTN. Las competencias específicas "
            "incluyen CE1.1 (especificar y desarrollar sistemas de información), "
            "CE1.3 (desarrollar software para soluciones informáticas), CE4.1 "
            "(certificar funcionamiento de sistemas) y CE5.1 (dirigir implementación "
            "de sistemas). Las competencias genéricas incluyen CG4 (uso de técnicas "
            "y herramientas), CG5 (generar desarrollos tecnológicos), CG8 (ética y "
            "responsabilidad profesional) y CG9 (aprendizaje continuo)."
        ),
    },
    {
        "id": "doc_evaluacion",
        "titulo": "Regularidad, promoción directa y asistencia",
        "contenido": (
            "Para regularizar el cursado de Inteligencia Artificial el estudiante "
            "debe: (a) aprobar los dos trabajos prácticos con un puntaje igual o "
            "superior a 4, (b) aprobar el parcial con un puntaje igual o superior "
            "a 4, y (c) realizar las actividades de evaluación continua y participar "
            "en el debate sobre ética en IA. Adicionalmente se requiere el 80 por "
            "ciento de asistencia. Las condiciones para la promoción directa son "
            "las mismas, pero los puntajes mínimos suben a 6 tanto en los dos "
            "trabajos prácticos como en el parcial; también se requiere haber "
            "realizado las actividades de evaluación continua y haber participado "
            "del debate, manteniendo el 80 por ciento de asistencia. La nota final "
            "es el promedio de las notas de los trabajos prácticos, las evaluaciones "
            "continuas y el parcial. Los alumnos que demuestren niveles mínimos y "
            "básicos de aprendizaje pero no hayan alcanzado la aprobación directa "
            "rendirán un examen final escrito que abarca todos los temas del "
            "programa."
        ),
    },
    {
        "id": "doc_tps",
        "titulo": "Trabajos prácticos y modalidad de agrupamiento",
        "contenido": (
            "La materia tiene dos trabajos prácticos. El TP1 consiste en diseñar un "
            "clasificador neuronal y se realiza en conjunto con la cátedra de "
            "Ciencia de Datos. El TP2 consiste en diseñar un agente con temática "
            "libre y se divide en tres etapas: la primera etapa es la definición "
            "del agente y los objetivos de diseño; la segunda etapa es la "
            "definición de percepciones y acciones del agente más la descripción "
            "del ambiente donde actúa, junto con implementaciones parciales; la "
            "tercera etapa es la entrega final con código, presentación e informe. "
            "Los trabajos prácticos y las actividades en clase se realizan en forma "
            "grupal. Los grupos están formados por 2 o 3 integrantes — es decir, "
            "1 menor estricto que nro-integrantes menor o igual que 3. Cada grupo "
            "debe enviar por email su formación indicando nombre, apellido y "
            "dirección de email de todos los integrantes. Los aspectos a evaluar "
            "en cada TP son: el informe técnico y la presentación, la definición y "
            "alcance de los objetivos del TP, la presentación de la solución de "
            "software, la cantidad y calidad de las herramientas que se apliquen, "
            "la calidad y cantidad de pruebas de validación, y la expresividad en "
            "los coloquios de presentación de los prácticos."
        ),
    },
    {
        "id": "doc_cronograma",
        "titulo": "Programación de actividades 1er cuatrimestre 2026",
        "contenido": (
            "La programación de actividades para el 1er cuatrimestre 2026 es la "
            "siguiente. El debate en equipos sobre ética en IA se realiza el 16 de "
            "marzo. El parcial tiene fecha 6 de abril y cubre el tema Búsqueda en "
            "espacio de estado, y se rinde en clase usando el campus virtual. El "
            "TP1 sobre el desarrollo del modelo neuronal tiene fecha de entrega "
            "el 18 de mayo. El TP2 se desarrolla en tres etapas con entregas "
            "sucesivas: la primera etapa, definición del agente y objetivos de "
            "diseño, vence el 8 de junio; la segunda etapa, definición de "
            "percepciones y acciones del agente con descripción del ambiente e "
            "implementaciones parciales, vence el 22 de junio; la entrega final "
            "con código, presentación e informe vence el 29 de junio. Las clases "
            "del cuatrimestre se dictan los lunes de 18 a 21 horas en el "
            "Laboratorio 1."
        ),
    },
    {
        "id": "doc_recursos",
        "titulo": "Bibliografía, software y canal de comunicación",
        "contenido": (
            "La bibliografía principal de la materia incluye: Inteligencia "
            "Artificial Un enfoque moderno de Russell y Norvig, disponible en "
            "aima.cs.berkeley.edu; Inteligencia Artificial Segunda Edición de Rich "
            "y Knight; Inteligencia Artificial Una nueva síntesis de Nilsson; "
            "MultiAgent Systems de Wooldridge; y los apuntes de clase elaborados "
            "por la cátedra. El software utilizado incluye Python como lenguaje "
            "principal, notebooks Jupyter para los desarrollos y Google Colab "
            "como entorno de ejecución en la nube. El canal de comunicación oficial "
            "entre estudiantes y docentes es el campus virtual de la FRSF, "
            "disponible en https://campusvirtual.frsf.utn.edu.ar/, donde se "
            "publican los materiales, se administran los cuestionarios de "
            "evaluación continua y se entregan los trabajos prácticos. Los "
            "objetivos pedagógicos generales de la cátedra son: gestionar proyectos "
            "de construcción de sistemas inteligentes, reconocer estrategias de "
            "creación de sistemas inteligentes, resolver problemas de "
            "representación del conocimiento y razonamiento en ambientes "
            "deterministas y bajo incertidumbre, y evaluar e implementar modelos "
            "de aprendizaje automático para resolver problemas."
        ),
    },
]

print(f'✓ {len(DOCUMENTOS)} documentos de la cátedra IA cargados')
for doc in DOCUMENTOS:
    print(f'  - {doc["titulo"]} ({len(doc["contenido"])} chars)')


## 3. Chunking

Partimos cada documento en **oraciones** (split por `. ! ?`). Es lo más simple y funciona bien para textos de reglamento donde cada oración es autocontenida.


In [ ]:
import re


def chunk_por_oracion(texto):
    """Chunking por oración (split en punto/exclamación/interrogación + espacio)."""
    oraciones = re.split(r'(?<=[.!?])\s+', texto)
    return [o.strip() for o in oraciones if o.strip()]


all_chunks = []
all_ids = []
all_metadatas = []
for doc in DOCUMENTOS:
    chunks = chunk_por_oracion(doc['contenido'])
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_ids.append(f'{doc["id"]}_chunk_{i}')
        all_metadatas.append({
            'titulo': doc['titulo'],
            'doc_id': doc['id'],
            'chunk_index': i,
        })

print(f'✓ {len(all_chunks)} chunks (oraciones) listas para indexar')
print(f'\nEjemplos:')
for chunk in all_chunks[:3]:
    print(f'  - "{chunk[:80]}..." ({len(chunk)} chars)')


## 4. Embeddings + ChromaDB (indexing)

Generamos un embedding por chunk y los almacenamos en ChromaDB in-memory.


In [ ]:
import chromadb

client = chromadb.Client()
try:
    client.delete_collection('clase3_docs')
except Exception:
    pass

collection = client.create_collection(
    name='clase3_docs',
    metadata={'hnsw:space': 'cosine'},
)

all_embeddings = modelo_emb.encode(all_chunks).tolist()

collection.add(
    documents=all_chunks,
    embeddings=all_embeddings,
    metadatas=all_metadatas,
    ids=all_ids,
)
print(f'✓ Indexados {collection.count()} chunks en ChromaDB')


## 5. Retrieval

Dada una query, buscamos los top-k chunks más similares por coseno.


In [ ]:
def buscar_chunks(query, n_results=3):
    """Retrieval denso: top-k por similitud coseno."""
    query_embedding = modelo_emb.encode([query]).tolist()
    return collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=['documents', 'metadatas', 'distances'],
    )


query = '¿Quiénes son los docentes de la cátedra?'
results = buscar_chunks(query, n_results=3)
print(f'Query: "{query}"\n\nTop 3 chunks:')
for i in range(len(results['documents'][0])):
    titulo = results['metadatas'][0][i]['titulo']
    sim = 1 - results['distances'][0][i]
    print(f'  #{i+1} (sim {sim:.3f}) [{titulo}]: "{results["documents"][0][i][:80]}..."')


## 6. RAG end-to-end — `rag_query`

System prompt fuerza al LLM a **basarse SOLO en el contexto** y **citar fuentes**.


In [ ]:
SYSTEM_RAG = """Sos un asistente de la cátedra Inteligencia Artificial de UTN-FRSF
que responde preguntas basándose ÚNICAMENTE en el contexto proporcionado.

Reglas:
- Usá SOLO la información del contexto. Nunca inventes fechas, nombres o números.
- Si el contexto no tiene información suficiente, decí "No tengo información suficiente en los documentos proporcionados."
- Citá la fuente entre corchetes cuando sea posible, ej: [Cronograma 2026].
- Respondé en español, conciso (máximo 5 oraciones)."""


def rag_query(pregunta, n_chunks=3, verbose=False):
    """Pipeline RAG completo: retrieval → augmentation → generation."""
    results = buscar_chunks(pregunta, n_results=n_chunks)

    contexto_partes = []
    for i in range(len(results['documents'][0])):
        doc = results['documents'][0][i]
        titulo = results['metadatas'][0][i]['titulo']
        contexto_partes.append(f'[{i+1}] ({titulo}): {doc}')
    contexto = '\n\n'.join(contexto_partes)

    messages = [
        {'role': 'system', 'content': SYSTEM_RAG},
        {'role': 'user', 'content': f'Contexto:\n{contexto}\n\nPregunta: {pregunta}'},
    ]
    respuesta = llamar_llm(messages, temperature=0.3)

    if verbose:
        print(f'Pregunta: {pregunta}\n')
        print(f'📎 Chunks recuperados:')
        for parte in contexto_partes:
            print(f'  {parte[:100]}...')
        print(f'\n💬 Respuesta:\n{respuesta}')

    return respuesta, results


respuesta, _ = rag_query('¿Cuáles son las fechas de entrega del TP2?', verbose=True)


# Parte 3 — Benchmark + correr RAG Naive

Definimos un benchmark de 7 queries sobre el programa, y lo corremos en paralelo por **LLM puro** y **RAG naive** para ver el contraste en datos.


## 7. El benchmark — 7 queries sobre el programa de la cátedra

Diseño deliberado para cubrir 4 tipos de query (fácil / ambigua-sigla / ambigua-concepto / multi-hop / edge case). Cada query golpea **datos específicos** que el LLM puro **no puede inventar correctamente**.


In [ ]:
BENCHMARK = [
    # ── Fáciles ──────────────────────────────────────────────────
    {
        'id': 'easy-1', 'tipo': 'easy',
        'pregunta': '¿Quiénes son los docentes de la cátedra Inteligencia Artificial en UTN-FRSF y qué cargo tiene cada uno?',
        'expected_keywords': ['Milagros Gutiérrez', 'Jorge Roa', 'Titular', 'Adjunto'],
        'expected_doc': 'doc_identidad',
    },
    {
        'id': 'easy-2', 'tipo': 'easy',
        'pregunta': '¿Qué día, horario y aula se dicta la materia?',
        'expected_keywords': ['lunes', '18', '21', 'Laboratorio 1', 'presencial'],
        'expected_doc': 'doc_identidad',
    },
    # ── Ambigua: sigla local ─────────────────────────────────────
    {
        'id': 'amb-1', 'tipo': 'ambigua-sigla',
        'pregunta': '¿Qué es el CIDISI y qué establece la Ordenanza 1877?',
        'expected_keywords': ['Centro de Investigación', 'Sistemas de Información', 'competencias', 'CE', 'CG'],
        'expected_doc': 'doc_identidad',
    },
    # ── Ambigua: concepto amplio ─────────────────────────────────
    {
        'id': 'amb-2', 'tipo': 'ambigua-concepto',
        'pregunta': '¿Cómo está organizado el sistema de evaluación de la materia?',
        'expected_keywords': ['trabajos prácticos', 'parcial', 'evaluación continua', 'debate', 'asistencia'],
        'expected_doc': 'doc_evaluacion',
    },
    # ── Multi-hop ────────────────────────────────────────────────
    {
        'id': 'multi-1', 'tipo': 'multi-hop',
        'pregunta': '¿Qué necesito hacer para promocionar la materia y qué fechas tengo que cumplir?',
        'expected_keywords': ['6', 'trabajos prácticos', 'parcial', '80', 'asistencia', '16', 'marzo', 'abril', '18', 'mayo', '29', 'junio'],
        'expected_docs': ['doc_evaluacion', 'doc_cronograma'],
    },
    {
        'id': 'multi-2', 'tipo': 'multi-hop',
        'pregunta': '¿Cuál es la diferencia entre regularizar y promocionar la materia?',
        'expected_keywords': ['4', '6', 'trabajos prácticos', 'parcial', '80', 'asistencia'],
        'expected_docs': ['doc_evaluacion'],
    },
    # ── Edge case ────────────────────────────────────────────────
    {
        'id': 'edge-1', 'tipo': 'edge-case',
        'pregunta': '¿Cuándo es la entrega del TP3?',
        'expected_keywords': ['no', 'sólo', 'TP1', 'TP2', 'dos'],
        'expected_doc': None,  # NO hay TP3 — el sistema debe abstenerse
    },
]

print(f'✓ Benchmark con {len(BENCHMARK)} queries')
for q in BENCHMARK:
    print(f'  [{q["tipo"]:18}] {q["pregunta"][:60]}')


## 8. ✏️ Ejercicio — side-by-side LLM puro vs RAG naive (10 min)

Corremos las 7 queries en **paralelo** por LLM puro y RAG. La métrica `score_keywords` cuenta cuántos de los hechos específicos del corpus aparecen en cada respuesta. **El LLM puro va a sacar casi 0 en los datos específicos** porque no puede saberlos.


In [ ]:
import pandas as pd


def side_by_side(benchmark, n_chunks=3):
    """Corre cada query por LLM puro y RAG naive."""
    rows = []
    for q in benchmark:
        resp_llm = llamar_llm([
            {'role': 'system', 'content': 'Respondé en español, máximo 4 oraciones.'},
            {'role': 'user', 'content': q['pregunta']},
        ], temperature=0.3)
        score_llm = score_keywords(resp_llm, q['expected_keywords'])

        resp_rag, results = rag_query(q['pregunta'], n_chunks=n_chunks, verbose=False)
        score_rag = score_keywords(resp_rag, q['expected_keywords'])
        docs_rag = [m['doc_id'] for m in results['metadatas'][0]]
        doc_ok = (q.get('expected_doc') in docs_rag) if q.get('expected_doc') else None

        rows.append({
            'id': q['id'],
            'tipo': q['tipo'],
            'pregunta': q['pregunta'][:45] + '...',
            'llm_kw': f'{score_llm}/{len(q["expected_keywords"])}',
            'rag_kw': f'{score_rag}/{len(q["expected_keywords"])}',
            'rag_doc_ok': '✓' if doc_ok else ('✗' if doc_ok is False else 'n/a'),
            'resp_llm': resp_llm,
            'resp_rag': resp_rag,
        })
    return pd.DataFrame(rows)


print('Corriendo benchmark (puede tardar 1-2 min)...\n')
df_sxs = side_by_side(BENCHMARK)
print(df_sxs[['id', 'tipo', 'pregunta', 'llm_kw', 'rag_kw', 'rag_doc_ok']].to_string(index=False))


### Inspección detallada — ¿qué inventó el LLM puro?

Mirá las respuestas. **Buscá fechas, nombres y umbrales** en la del LLM puro: si no están en los hechos correctos del corpus, **son inventos**.


In [ ]:
for _, row in df_sxs.iterrows():
    print('═' * 70)
    print(f'[{row["tipo"]}] {row["id"]} — {row["pregunta"]}')
    print(f'\n  LLM puro ({row["llm_kw"]}):')
    print(f'    {row["resp_llm"][:300]}')
    print(f'\n  RAG naive ({row["rag_kw"]}, doc_ok={row["rag_doc_ok"]}):')
    print(f'    {row["resp_rag"][:300]}')
    print()


## 9. Visualización — el contraste LLM puro vs RAG naive


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def to_frac(s):
    n, m = s.split('/')
    return int(n) / int(m) if int(m) > 0 else 0


df_sxs['llm_frac'] = df_sxs['llm_kw'].apply(to_frac)
df_sxs['rag_frac'] = df_sxs['rag_kw'].apply(to_frac)

by_tipo = df_sxs.groupby('tipo')[['llm_frac', 'rag_frac']].mean()

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(by_tipo))
w = 0.35
ax.bar(x - w/2, by_tipo['llm_frac'], w, label='LLM puro', color='#E07474')
ax.bar(x + w/2, by_tipo['rag_frac'], w, label='RAG naive', color='#5BC85B')
ax.set_xticks(x)
ax.set_xticklabels(by_tipo.index, rotation=20, ha='right')
ax.set_ylabel('Fracción de keywords cubiertos (avg)')
ax.set_title('LLM puro vs RAG naive — score promedio por tipo de query')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)
for i, (lv, rv) in enumerate(zip(by_tipo['llm_frac'], by_tipo['rag_frac'])):
    ax.text(i - w/2, lv + 0.02, f'{lv:.0%}', ha='center', fontsize=9)
    ax.text(i + w/2, rv + 0.02, f'{rv:.0%}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print()
print(f'Score total LLM puro:  {df_sxs["llm_frac"].mean():.0%}')
print(f'Score total RAG naive: {df_sxs["rag_frac"].mean():.0%}')


### Reflexión rápida

1. **El LLM puro no puede saber datos específicos** (fechas, cargos, ordenanzas). Si menciona alguno, lo inventa.
2. **RAG naive ya gana en datos específicos**, pero todavía falla en:
   - **Multi-hop** (combinar 2 documentos)
   - **Edge case** (TP3 no existe — depende de si el LLM se abstiene)
3. Las próximas técnicas (BM25, reranking, HyDE) van a atacar esos casos.


# Parte 4 — Hybrid Search (BM25 + dense)

El RAG naive usa **sólo dense retrieval** (embeddings). Esto falla cuando la query contiene siglas o términos muy específicos (ej. "Ord. 1877", "ISI", "CIDISI"). **BM25** (búsqueda léxica) acierta esos casos. **Hybrid** combina ambos.


## 10. BM25 standalone


In [ ]:
from rank_bm25 import BM25Okapi


def tokenize_simple(text):
    return re.findall(r'\w+', text.lower())


corpus_tok = [tokenize_simple(c) for c in all_chunks]
bm25 = BM25Okapi(corpus_tok)


def buscar_bm25(query, n_results=3):
    qt = tokenize_simple(query)
    scores = bm25.get_scores(qt)
    top = np.argsort(scores)[::-1][:n_results]
    return [
        {'chunk': all_chunks[i], 'score': float(scores[i]),
         'metadata': all_metadatas[i], 'idx': int(i)}
        for i in top
    ]


# Demo: la sigla "Ordenanza 1877"
query = 'Ordenanza 1877'
print(f'Query: "{query}"\n')
print('── BM25 (léxico): ──')
for i, r in enumerate(buscar_bm25(query)):
    print(f'  #{i+1} (score {r["score"]:.2f}) [{r["metadata"]["titulo"]}]:')
    print(f'       "{r["chunk"][:90]}..."')

print('\n── Dense (ChromaDB): ──')
sem = buscar_chunks(query)
for i in range(3):
    sim = 1 - sem['distances'][0][i]
    print(f'  #{i+1} (sim {sim:.3f}) [{sem["metadatas"][0][i]["titulo"]}]:')
    print(f'       "{sem["documents"][0][i][:90]}..."')

print('\n💡 Para una sigla como "Ord. 1877", BM25 acierta el match exacto.')


## 11. Hybrid Search — combinar BM25 + dense

`score_final = α · score_dense_norm + (1-α) · score_bm25_norm`
- α=1.0 → solo dense
- α=0.0 → solo BM25
- α=0.5 → balanceado


In [ ]:
def hybrid_search(query, n_results=3, alpha=0.5):
    """Hybrid: weighted sum de BM25 + dense, ambos normalizados a [0,1]."""
    qt = tokenize_simple(query)
    bs = bm25.get_scores(qt)
    bn = bs / bs.max() if bs.max() > 0 else bs

    qe = modelo_emb.encode([query]).tolist()
    r = collection.query(query_embeddings=qe, n_results=len(all_chunks), include=['distances'])
    sm = np.zeros(len(all_chunks))
    for i, idx_id in enumerate(r['ids'][0]):
        sm[all_ids.index(idx_id)] = 1 - r['distances'][0][i]
    sn = sm / sm.max() if sm.max() > 0 else sm

    h = alpha * sn + (1 - alpha) * bn
    top = np.argsort(h)[::-1][:n_results]
    return [
        {'chunk': all_chunks[i], 'hybrid_score': float(h[i]),
         'sem_score': float(sn[i]), 'bm25_score': float(bn[i]),
         'metadata': all_metadatas[i], 'idx': int(i)}
        for i in top
    ]


# Demo
query = '¿Cómo se aprueba la materia?'
print(f'Query: "{query}"\n')
print('── Hybrid (α=0.5): ──')
for i, r in enumerate(hybrid_search(query, alpha=0.5)):
    print(f'  #{i+1} (h={r["hybrid_score"]:.3f}|sem={r["sem_score"]:.3f}|bm25={r["bm25_score"]:.3f})')
    print(f'       "{r["chunk"][:80]}..."')


## 12. RAG con selección de retriever

Adaptamos `rag_query` para que reciba `retriever={'dense', 'bm25', 'hybrid'}`.


In [ ]:
def rag_query_with_retriever(pregunta, retriever='dense', n_chunks=3, alpha=0.5):
    if retriever == 'dense':
        r = buscar_chunks(pregunta, n_results=n_chunks)
        docs = r['documents'][0]
        metas = r['metadatas'][0]
    elif retriever == 'bm25':
        br = buscar_bm25(pregunta, n_results=n_chunks)
        docs = [x['chunk'] for x in br]
        metas = [x['metadata'] for x in br]
    elif retriever == 'hybrid':
        hr = hybrid_search(pregunta, n_results=n_chunks, alpha=alpha)
        docs = [x['chunk'] for x in hr]
        metas = [x['metadata'] for x in hr]
    else:
        raise ValueError(retriever)

    contexto = '\n\n'.join(
        f'[{i+1}] ({metas[i]["titulo"]}): {docs[i]}'
        for i in range(len(docs))
    )
    messages = [
        {'role': 'system', 'content': SYSTEM_RAG},
        {'role': 'user', 'content': f'Contexto:\n{contexto}\n\nPregunta: {pregunta}'},
    ]
    respuesta = llamar_llm(messages, temperature=0.3)
    return respuesta, [m['doc_id'] for m in metas]


# Smoke
respuesta, docs = rag_query_with_retriever('¿Qué es el CIDISI?', retriever='bm25')
print(f'[bm25] docs: {docs}')
print(f'[bm25] resp: {respuesta[:250]}')


## 13. ✏️ Ejercicio — corré el benchmark con los 3 métodos (8 min)


In [ ]:
def comparar_3_metodos(benchmark, n_chunks=3, alpha=0.5):
    rows = []
    for q in benchmark:
        row = {'id': q['id'], 'tipo': q['tipo'], 'pregunta': q['pregunta'][:40] + '...'}
        for metodo in ['dense', 'bm25', 'hybrid']:
            respuesta, docs = rag_query_with_retriever(
                q['pregunta'], retriever=metodo, n_chunks=n_chunks, alpha=alpha,
            )
            kw = score_keywords(respuesta, q['expected_keywords'])
            row[f'{metodo}'] = f'{kw}/{len(q["expected_keywords"])}'
        rows.append(row)
    return pd.DataFrame(rows)


print('Corriendo benchmark con los 3 métodos (~1.5 min)...\n')
df_compare = comparar_3_metodos(BENCHMARK)
print(df_compare.to_string(index=False))


## 14. Visualización — comparativa de los 3 métodos


In [ ]:
for col in ['dense', 'bm25', 'hybrid']:
    df_compare[f'{col}_f'] = df_compare[col].apply(to_frac)

by_tipo = df_compare.groupby('tipo')[['dense_f', 'bm25_f', 'hybrid_f']].mean()

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(by_tipo))
w = 0.27
ax.bar(x - w, by_tipo['dense_f'], w, label='Dense', color='#5A95E0')
ax.bar(x,     by_tipo['bm25_f'],  w, label='BM25',  color='#E0A04D')
ax.bar(x + w, by_tipo['hybrid_f'], w, label='Hybrid', color='#5BC85B')
ax.set_xticks(x)
ax.set_xticklabels(by_tipo.index, rotation=20, ha='right')
ax.set_ylabel('Fracción de keywords cubiertos (avg)')
ax.set_title('Dense vs BM25 vs Hybrid — score promedio por tipo de query')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nScores globales:')
print(f'  Dense:  {df_compare["dense_f"].mean():.0%}')
print(f'  BM25:   {df_compare["bm25_f"].mean():.0%}')
print(f'  Hybrid: {df_compare["hybrid_f"].mean():.0%}')


# Parte 5 — Técnicas avanzadas: Reranking + HyDE + Parent-Child

Sumamos tres técnicas que atacan distintos problemas del retrieval naive.


## 15. Reranking con cross-encoder

**Idea:** retrieval amplio (top-N) seguido de un cross-encoder que re-evalúa cada par (query, chunk) con más precisión.


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')


def buscar_con_reranking(query, n_initial=10, n_final=3):
    initial = buscar_chunks(query, n_results=n_initial)
    cand = initial['documents'][0]
    metas = initial['metadatas'][0]
    pares = [[query, doc] for doc in cand]
    scores = reranker.predict(pares)
    ranking = np.argsort(scores)[::-1][:n_final]
    return [
        {'chunk': cand[i], 'score': float(scores[i]), 'metadata': metas[i]}
        for i in ranking
    ]


query = '¿Qué necesito hacer para promocionar la materia y qué fechas tengo que cumplir?'
print(f'Query: "{query}"\n')

print('── Sin reranking (top-3 dense): ──')
sem = buscar_chunks(query, n_results=3)
for i in range(3):
    sim = 1 - sem['distances'][0][i]
    print(f'  #{i+1} (sim {sim:.3f}) [{sem["metadatas"][0][i]["titulo"]}]')

print('\n── Con reranking (initial=10 → top-3): ──')
for i, r in enumerate(buscar_con_reranking(query)):
    print(f'  #{i+1} (rerank {r["score"]:+.3f}) [{r["metadata"]["titulo"]}]')


## 16. HyDE — Hypothetical Document Embeddings

**Idea:** la query es corta, los chunks son largos. Pedile al LLM que **invente** un documento que respondería la pregunta, y buscá con su embedding.


In [ ]:
def hyde_search(query, n_results=3):
    messages = [
        {'role': 'system', 'content':
            'Escribí un párrafo técnico de 3-4 oraciones que responda la pregunta. '
            'No inventes datos específicos. Escribí como si fuera un fragmento de '
            'reglamento académico o documento institucional.'},
        {'role': 'user', 'content': query},
    ]
    doc_hip = llamar_llm(messages, temperature=0.5)
    he = modelo_emb.encode([doc_hip]).tolist()
    results = collection.query(
        query_embeddings=he, n_results=n_results,
        include=['documents', 'metadatas', 'distances'],
    )
    return results, doc_hip


query = '¿Qué herramientas usa la cátedra?'
hr, doc_hip = hyde_search(query)
print(f'Query: "{query}"\n')
print(f'── Doc hipotético generado: ──')
print(f'  "{doc_hip[:250]}..."\n')
print(f'── Búsqueda HyDE: ──')
for i in range(3):
    sim = 1 - hr['distances'][0][i]
    print(f'  #{i+1} (sim {sim:.3f}) [{hr["metadatas"][0][i]["titulo"]}]: "{hr["documents"][0][i][:80]}..."')


## 17. Parent-Child chunks

**Idea:** matcheamos por oración (chunks chicos, retrieval preciso) pero al LLM le pasamos el documento entero (parent, contexto rico).


In [ ]:
parent_text_by_doc = {d['id']: d['contenido'] for d in DOCUMENTOS}


def buscar_parent_child(query, n_results=3, expand=True):
    r = buscar_chunks(query, n_results=n_results*2)
    if expand:
        seen, out = set(), []
        for i in range(len(r['documents'][0])):
            did = r['metadatas'][0][i]['doc_id']
            if did not in seen and len(out) < n_results:
                seen.add(did)
                out.append({
                    'parent_text': parent_text_by_doc[did],
                    'titulo': r['metadatas'][0][i]['titulo'],
                    'doc_id': did,
                    'matched_chunk': r['documents'][0][i],
                    'sim': 1 - r['distances'][0][i],
                })
        return out
    return [
        {'parent_text': r['documents'][0][i],
         'titulo': r['metadatas'][0][i]['titulo'],
         'doc_id': r['metadatas'][0][i]['doc_id'],
         'matched_chunk': r['documents'][0][i],
         'sim': 1 - r['distances'][0][i]}
        for i in range(min(n_results, len(r['documents'][0])))
    ]


query = '¿Cuándo se entrega el TP1?'
print(f'Query: "{query}"\n')

print('── Sin expand (sólo oración matcheada): ──')
for i, r in enumerate(buscar_parent_child(query, expand=False)):
    print(f'  #{i+1} [{r["titulo"]}]: "{r["matched_chunk"][:80]}..."')

print('\n── Con expand (devuelve el doc entero): ──')
for i, r in enumerate(buscar_parent_child(query, expand=True)):
    print(f'  #{i+1} [{r["titulo"]}] (parent {len(r["parent_text"])} chars)')
    print(f'      Matched: "{r["matched_chunk"][:60]}..."')


## 18. `rag_query_advanced` — modo por modo


In [ ]:
def rag_query_advanced(pregunta, modo='reranking', n_chunks=3):
    if modo == 'baseline':
        r = buscar_chunks(pregunta, n_results=n_chunks)
        docs = r['documents'][0]
        metas = r['metadatas'][0]

    elif modo == 'reranking':
        rr = buscar_con_reranking(pregunta, n_initial=10, n_final=n_chunks)
        docs = [x['chunk'] for x in rr]
        metas = [x['metadata'] for x in rr]

    elif modo == 'hyde':
        hr, _ = hyde_search(pregunta, n_results=n_chunks)
        docs = hr['documents'][0]
        metas = hr['metadatas'][0]

    elif modo == 'parent':
        pc = buscar_parent_child(pregunta, n_results=n_chunks, expand=True)
        docs = [x['parent_text'] for x in pc]
        metas = [{'titulo': x['titulo'], 'doc_id': x['doc_id']} for x in pc]

    elif modo == 'all':
        # HyDE → retrieval ampliado → rerank → parent expand
        _, doc_hip = hyde_search(pregunta, n_results=1)
        he = modelo_emb.encode([doc_hip]).tolist()
        r = collection.query(query_embeddings=he, n_results=10, include=['documents', 'metadatas'])
        cand = r['documents'][0]
        meta_i = r['metadatas'][0]
        pares = [[pregunta, d] for d in cand]
        scores = reranker.predict(pares)
        ranking = np.argsort(scores)[::-1][:n_chunks*2]
        seen, docs, metas = set(), [], []
        for idx in ranking:
            did = meta_i[idx]['doc_id']
            if did not in seen and len(docs) < n_chunks:
                seen.add(did)
                docs.append(parent_text_by_doc[did])
                metas.append({'titulo': meta_i[idx]['titulo'], 'doc_id': did})
    else:
        raise ValueError(modo)

    contexto = '\n\n'.join(
        f'[{i+1}] ({metas[i]["titulo"]}): {docs[i]}'
        for i in range(len(docs))
    )
    respuesta = llamar_llm([
        {'role': 'system', 'content': SYSTEM_RAG},
        {'role': 'user', 'content': f'Contexto:\n{contexto}\n\nPregunta: {pregunta}'},
    ], temperature=0.3)
    return respuesta, [m['doc_id'] for m in metas]


## 19. ✏️ Ejercicio — corré el benchmark con los 5 modos (10 min)


In [ ]:
def comparar_avanzado(benchmark, n_chunks=3):
    rows = []
    for q in benchmark:
        row = {'id': q['id'], 'tipo': q['tipo'], 'pregunta': q['pregunta'][:40] + '...'}
        for modo in ['baseline', 'reranking', 'hyde', 'parent', 'all']:
            respuesta, docs = rag_query_advanced(q['pregunta'], modo=modo, n_chunks=n_chunks)
            kw = score_keywords(respuesta, q['expected_keywords'])
            row[f'{modo}'] = f'{kw}/{len(q["expected_keywords"])}'
        rows.append(row)
    return pd.DataFrame(rows)


print('Corriendo benchmark con los 5 modos (~3 min, paciencia)...\n')
df_avanzado = comparar_avanzado(BENCHMARK)
print(df_avanzado.to_string(index=False))


## 20. Visualización — score global por modo


In [ ]:
modos = ['baseline', 'reranking', 'hyde', 'parent', 'all']
for col in modos:
    df_avanzado[f'{col}_f'] = df_avanzado[col].apply(to_frac)
scores_globales = [df_avanzado[f'{m}_f'].mean() for m in modos]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#999', '#5A95E0', '#E0A04D', '#4DA6A6', '#5BC85B']
bars = ax.bar(modos, scores_globales, color=colors)
ax.set_ylabel('Score promedio (fracción de keywords)')
ax.set_title('Modos de RAG — score global sobre el benchmark de la cátedra')
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)
for bar, s in zip(bars, scores_globales):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{s:.0%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


# Parte 6 — Capstone: evolución de una query difícil

Tomamos UNA query multi-hop y la hacemos pasar por los 6 stages que vimos. Vemos la evolución del score como un gráfico de barras.


## 21. La query elegida y el ground truth

> *"¿Qué necesito aprobar para promocionar la materia y cuándo son las entregas más importantes del cuatrimestre?"*

Es **multi-hop** (combina `doc_evaluacion` + `doc_cronograma` + `doc_tps`) y contiene datos que el LLM puro no puede inventar correctamente.


In [ ]:
QUERY = (
    '¿Qué necesito aprobar para promocionar la materia Inteligencia Artificial '
    'y cuándo son las entregas más importantes del cuatrimestre?'
)

GROUND_TRUTH_KEYWORDS = [
    # de doc_evaluacion
    '6', 'trabajos prácticos', 'parcial', '80', 'asistencia', 'evaluación continua', 'debate',
    # de doc_cronograma
    '16', 'marzo', 'abril', '18', 'mayo', '8', 'junio', '29',
    # de doc_tps
    'coloquios',
]

DOCS_ESPERADOS = ['doc_evaluacion', 'doc_cronograma', 'doc_tps']
print(f'{len(GROUND_TRUTH_KEYWORDS)} keywords esperados; docs esperados: {DOCS_ESPERADOS}')


## Stage 1 — LLM puro


In [ ]:
resp_puro = llamar_llm([
    {'role': 'system', 'content':
        'Sos un asistente de la cátedra IA de UTN-FRSF. Respondé técnicamente, en español, máximo 6 oraciones.'},
    {'role': 'user', 'content': QUERY},
], temperature=0.3)
score_puro = score_keywords(resp_puro, GROUND_TRUTH_KEYWORDS)
print(f'STAGE 1 — Score: {score_puro}/{len(GROUND_TRUTH_KEYWORDS)}\n\n{resp_puro}')


## Stages 2 a 6 — RAG con técnicas progresivas


In [ ]:
# Stage 2: RAG Naive
resp_naive, docs_naive = rag_query_with_retriever(QUERY, retriever='dense', n_chunks=3)
score_naive = score_keywords(resp_naive, GROUND_TRUTH_KEYWORDS)

# Stage 3: + Hybrid
resp_hybrid, docs_hybrid = rag_query_with_retriever(QUERY, retriever='hybrid', n_chunks=3)
score_hybrid = score_keywords(resp_hybrid, GROUND_TRUTH_KEYWORDS)

# Stage 4: + Reranking
resp_rerank, docs_rerank = rag_query_advanced(QUERY, modo='reranking', n_chunks=3)
score_rerank = score_keywords(resp_rerank, GROUND_TRUTH_KEYWORDS)

# Stage 5: + HyDE
resp_hyde, docs_hyde = rag_query_advanced(QUERY, modo='hyde', n_chunks=3)
score_hyde = score_keywords(resp_hyde, GROUND_TRUTH_KEYWORDS)

# Stage 6: + Parent-Child (all combinado)
resp_all, docs_all = rag_query_advanced(QUERY, modo='all', n_chunks=3)
score_all = score_keywords(resp_all, GROUND_TRUTH_KEYWORDS)

stages_data = [
    {'stage': '1. LLM puro',       'score': score_puro,   'docs': '(none)',                  'resp': resp_puro[:150]},
    {'stage': '2. RAG Naive',      'score': score_naive,  'docs': str(list(set(docs_naive))),  'resp': resp_naive[:150]},
    {'stage': '3. + Hybrid',       'score': score_hybrid, 'docs': str(list(set(docs_hybrid))), 'resp': resp_hybrid[:150]},
    {'stage': '4. + Reranking',    'score': score_rerank, 'docs': str(list(set(docs_rerank))), 'resp': resp_rerank[:150]},
    {'stage': '5. + HyDE',         'score': score_hyde,   'docs': str(list(set(docs_hyde))),   'resp': resp_hyde[:150]},
    {'stage': '6. + Parent (all)', 'score': score_all,    'docs': str(list(set(docs_all))),    'resp': resp_all[:150]},
]
df_evol = pd.DataFrame(stages_data)
print(df_evol[['stage', 'score', 'docs']].to_string(index=False))


## 24. Visualización — evolución del score por stage


In [ ]:
stages = [s['stage'] for s in stages_data]
scores = [s['score'] for s in stages_data]
max_score = len(GROUND_TRUTH_KEYWORDS)

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(stages, scores, color=['#999', '#7F77DD', '#5A95E0', '#4DA6A6', '#E0A04D', '#5BC85B'])
ax.axhline(max_score, color='red', linestyle='--', alpha=0.5, label=f'Máximo posible ({max_score})')
ax.set_ylabel(f'Keywords cubiertos / {max_score}')
ax.set_title(f'Evolución del score por stage — Cátedra IA UTN-FRSF\nQuery: "{QUERY[:60]}..."')
ax.set_ylim(0, max_score + 1)
for bar, s in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(s), ha='center', fontweight='bold')
plt.xticks(rotation=15, ha='right')
ax.legend()
plt.tight_layout()
plt.show()


## 25. Conclusión

**Lo que viste:**

1. **LLM puro** da una respuesta vaga o **inventa fechas y umbrales** con confianza.
2. **RAG Naive** ya cubre datos específicos pero tiende a quedarse en 1 documento → para multi-hop le falta cobertura.
3. **Hybrid** ayuda cuando hay siglas; en queries conceptuales el aporte es marginal.
4. **Reranking** sube chunks borderline al top y mejora cobertura entre docs.
5. **HyDE** convierte la query corta en un texto que "habla" como el corpus → empuja recall.
6. **Parent-child + all combinado** le da al LLM contexto completo de cada doc relevante.

**Tradeoff de costo/latencia:**

| Stage | Latencia adicional | Costo |
|---|---|---|
| 1 | baseline | 1 llamada LLM |
| 2 | + embed (30ms) | retrieval gratis (local) |
| 3 | + BM25 (10ms) | gratis |
| 4 | + reranker (50ms) | gratis (local) |
| 5 | **+ llamada LLM extra** | **+$$$ por query** |
| 6 | (igual stage 5) | + más tokens en el prompt |

### El takeaway

No siempre querés stage 6. **Empezá por naive, agregá técnicas SOLO cuando una métrica concreta lo justifique** — eso lo medís en **Clase 3b**.
